# 01 — Data preparation: labels and patient-level splits

**Project:** Explainable classification of breast masses as benign or malignant under low-resource conditions
(MobileNetV2, CBIS-DDSM).

**What this notebook does.** It reads the official CBIS-DDSM case lists for **masses**, turns the pathology
into a binary label, checks the data, and splits it into **training, validation and test** sets so that no
patient appears in more than one set. No images are used here; the images are fetched in notebook 02.

**Pipeline**

| Notebook | Job |
|---|---|
| **01 Data preparation** (this one) | Labels and patient-level splits |
| 02 Patch extraction | Download original DICOM images, cut a patch and outline around each mass |
| 03 Training and calibration | Baseline and fine-tuned MobileNetV2; lock temperature and threshold on validation |
| 04 Testing and robustness | Test once on clean and degraded images; 8-bit low-resource model; speed |

**Input** (Kaggle: *Add Input*)

| File | Source |
|---|---|
| `mass_case_description_train_set.csv` | CBIS-DDSM, The Cancer Imaging Archive ("Mass-Training-Description") |
| `mass_case_description_test_set.csv` | CBIS-DDSM, The Cancer Imaging Archive ("Mass-Test-Description") |

Here they are attached from the Kaggle dataset `cbis-ddsm-mass-metadata`. The same two files can be
downloaded from the CBIS-DDSM page of The Cancer Imaging Archive.
Dataset citation: Sawyer-Lee, R., Gimenez, F., Hoogi, A., & Rubin, D. (2016). *Curated Breast Imaging Subset of
Digital Database for Screening Mammography (CBIS-DDSM)* (Version 1) [Data set]. The Cancer Imaging Archive.
https://doi.org/10.7937/K9/TCIA.2016.7O02S9CY

**Output** (`/kaggle/working/splits`): `train.csv`, `val.csv`, `test.csv`, the case lists with a `label` column.

**Settings.** No GPU and no internet needed.

## 1. Check the attached input files
Lists every file under `/kaggle/input`, where Kaggle places attached datasets, to confirm the two case lists are there.

In [1]:
import os
import pandas as pd

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/hameedah/cbis-ddsm-mass-metadata/mass_case_description_test_set.csv
/kaggle/input/datasets/hameedah/cbis-ddsm-mass-metadata/mass_case_description_train_set.csv


**Result:** both case lists are attached.

## 2. Load the official case lists
Finds the two CSV files by name and reads each into a table (one row per outlined mass).
The columns used later are:

| Column | Meaning |
|---|---|
| `patient_id` | Patient identifier, used to keep each patient in one set |
| `left or right breast`, `image view`, `abnormality id` | Which breast, which view (CC = top-down, MLO = angled side view), which mass on that image |
| `pathology` | MALIGNANT, BENIGN or BENIGN_WITHOUT_CALLBACK |
| `image file path` | The full mammogram |
| `cropped image file path`, `ROI mask file path` | The small supplied crop and the radiologist's outline of the mass |

The other columns (breast density, mass shape and margins, BI-RADS assessment, subtlety) are descriptive and not used by the model.

In [2]:
from pathlib import Path

files = list(Path("/kaggle/input").rglob("*.csv"))

train_path = next(
    p for p in files
    if p.name == "mass_case_description_train_set.csv"
)

test_path = next(
    p for p in files
    if p.name == "mass_case_description_test_set.csv"
)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training records:", len(train_df))
print("Test records:", len(test_df))

display(train_df.head())
display(test_df.head())

Training records: 1318
Test records: 378


,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00001,3,LEFT,CC,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
1,P_00001,3,LEFT,MLO,1,mass,IRREGULAR-ARCHITECTURAL_DISTORTION,SPICULATED,4,MALIGNANT,4,Mass-Training_P_00001_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00001_LEFT_MLO_1/1.3.6.1.4.1.9...
2,P_00004,3,LEFT,CC,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_CC/1.3.6.1.4.1.9590...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...,Mass-Training_P_00004_LEFT_CC_1/1.3.6.1.4.1.95...
3,P_00004,3,LEFT,MLO,1,mass,ARCHITECTURAL_DISTORTION,ILL_DEFINED,4,BENIGN,3,Mass-Training_P_00004_LEFT_MLO/1.3.6.1.4.1.959...,Mass-Training_P_00004_LEFT_MLO_1/1.3.6.1.4.1.9...,Mass-Training_P_00004_LEFT_MLO_1/1.3.6.1.4.1.9...
4,P_00004,3,RIGHT,MLO,1,mass,OVAL,CIRCUMSCRIBED,4,BENIGN,5,Mass-Training_P_00004_RIGHT_MLO/1.3.6.1.4.1.95...,Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1.4.1....,Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1.4.1....


,patient_id,breast_density,left or right breast,image view,abnormality id,abnormality type,mass shape,mass margins,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00016,4,LEFT,CC,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00016_LEFT_CC/1.3.6.1.4.1.9590.100...,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.1...,Mass-Test_P_00016_LEFT_CC_1/1.3.6.1.4.1.9590.1...
1,P_00016,4,LEFT,MLO,1,mass,IRREGULAR,SPICULATED,5,MALIGNANT,5,Mass-Test_P_00016_LEFT_MLO/1.3.6.1.4.1.9590.10...,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590....,Mass-Test_P_00016_LEFT_MLO_1/1.3.6.1.4.1.9590....
2,P_00017,2,LEFT,CC,1,mass,ROUND,CIRCUMSCRIBED,4,MALIGNANT,4,Mass-Test_P_00017_LEFT_CC/1.3.6.1.4.1.9590.100...,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.1...,Mass-Test_P_00017_LEFT_CC_1/1.3.6.1.4.1.9590.1...
3,P_00017,2,LEFT,MLO,1,mass,ROUND,ILL_DEFINED,4,MALIGNANT,4,Mass-Test_P_00017_LEFT_MLO/1.3.6.1.4.1.9590.10...,Mass-Test_P_00017_LEFT_MLO_1/1.3.6.1.4.1.9590....,Mass-Test_P_00017_LEFT_MLO_1/1.3.6.1.4.1.9590....
4,P_00032,3,RIGHT,CC,1,mass,ROUND,OBSCURED,0,BENIGN,2,Mass-Test_P_00032_RIGHT_CC/1.3.6.1.4.1.9590.10...,Mass-Test_P_00032_RIGHT_CC_1/1.3.6.1.4.1.9590....,Mass-Test_P_00032_RIGHT_CC_1/1.3.6.1.4.1.9590....


**Result:** 1,318 masses in the official training list and 378 in the official test list (1,696 in total).

## 3. Pathology labels in the official lists
Counts each pathology value. BENIGN_WITHOUT_CALLBACK means the mass was judged benign on imaging and the
patient was not recalled for biopsy.

In [3]:
print("TRAINING SET")
print(train_df["pathology"].value_counts())

print("\nTEST SET")
print(test_df["pathology"].value_counts())

TRAINING SET
pathology
MALIGNANT                  637
BENIGN                     577
BENIGN_WITHOUT_CALLBACK    104
Name: count, dtype: int64

TEST SET
pathology
BENIGN                     194
MALIGNANT                  147
BENIGN_WITHOUT_CALLBACK     37
Name: count, dtype: int64


**Result:**

| List | Malignant | Benign | Benign without callback |
|---|---|---|---|
| Official training | 637 | 577 | 104 |
| Official test | 147 | 194 | 37 |

## 4. Binary labels
**Decision:** malignant = 1; benign **and** benign-without-callback = 0. Keeping the benign-without-callback
masses uses all the data, but it means the labels are *pathology-verified or radiologist-assessed*, not all
biopsy-confirmed. `value_counts(dropna=False)` would show any row with an unexpected value; there are none.

In [5]:
def encode_pathology(value):
    if value in ["BENIGN", "BENIGN_WITHOUT_CALLBACK"]:
        return 0
    elif value == "MALIGNANT":
        return 1
    else:
        return None

train_df["label"] = train_df["pathology"].apply(encode_pathology)
test_df["label"] = test_df["pathology"].apply(encode_pathology)

print("Training labels:")
print(train_df["label"].value_counts(dropna=False))

print("\nTest labels:")
print(test_df["label"].value_counts(dropna=False))

Training labels:
label
0    681
1    637
Name: count, dtype: int64

Test labels:
label
0    231
1    147
Name: count, dtype: int64


**Result:** training 681 benign / 637 malignant; test 231 benign / 147 malignant. 141 benign-without-callback masses are counted as benign.

## 5. Patients per list
A patient usually has several rows: the same mass in two views (CC and MLO), or more than one mass.

In [6]:
print("Training patients:", train_df["patient_id"].nunique())
print("Test patients:", test_df["patient_id"].nunique())

Training patients: 691
Test patients: 201


**Result:** 691 patients in the official training list, 201 in the test list.

## 6. Check that no patient is in both official lists
If one patient's images were in both training and test, the test result would be inflated.

In [7]:
train_patients = set(train_df["patient_id"])
test_patients = set(test_df["patient_id"])

overlap = train_patients.intersection(test_patients)

print("Patients appearing in both datasets:", len(overlap))
print("Overlapping patient IDs:", sorted(overlap))

Patients appearing in both datasets: 0
Overlapping patient IDs: []


**Result:** 0 patients in both lists, so the official split is already separated by patient.

## 7. Missing values
Only the descriptive mass shape and mass margins columns have gaps. Neither is used by the model, so no rows are dropped.

In [8]:
print("TRAINING SET MISSING VALUES")
print(train_df.isnull().sum())

print("\nTEST SET MISSING VALUES")
print(test_df.isnull().sum())

TRAINING SET MISSING VALUES
patient_id                  0
breast_density              0
left or right breast        0
image view                  0
abnormality id              0
abnormality type            0
mass shape                  4
mass margins               43
assessment                  0
pathology                   0
subtlety                    0
image file path             0
cropped image file path     0
ROI mask file path          0
label                       0
dtype: int64

TEST SET MISSING VALUES
patient_id                  0
breast_density              0
left or right breast        0
image view                  0
abnormality id              0
abnormality type            0
mass shape                  0
mass margins               17
assessment                  0
pathology                   0
subtlety                    0
image file path             0
cropped image file path     0
ROI mask file path          0
label                       0
dtype: int64


**Result:** mass shape 4 missing (training); mass margins 43 (training) and 17 (test). Not used.

## 8. Inspect the file paths
Each path ends with `.../<series ID>/000000.dcm`. Notebook 02 takes the series ID from these paths to download the images.

In [9]:
columns = [
    "patient_id",
    "pathology",
    "image file path",
    "cropped image file path",
    "ROI mask file path"
]

print(train_df[columns].iloc[0].to_string())

patient_id                                                           P_00001
pathology                                                          MALIGNANT
image file path            Mass-Training_P_00001_LEFT_CC/1.3.6.1.4.1.9590...
cropped image file path    Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...
ROI mask file path         Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.95...


## 9. Validation set, split by patient
15% of the **patients** in the official training list are held out for validation, with all of a patient's
rows kept together (`GroupShuffleSplit`, `random_state=42` so the split is reproducible). The validation set
is used in notebook 03 to choose the stopping epoch, fit the temperature and set the threshold. The official
test list is left untouched as the test set.

In [10]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        groups=train_df["patient_id"]
    )
)

final_train_df = train_df.iloc[train_idx].copy()
val_df = train_df.iloc[val_idx].copy()

print("Training records:", len(final_train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))

print("\nUnique patients:")
print("Train:", final_train_df["patient_id"].nunique())
print("Validation:", val_df["patient_id"].nunique())
print("Test:", test_df["patient_id"].nunique())

Training records: 1121
Validation records: 197
Test records: 378

Unique patients:
Train: 587
Validation: 104
Test: 201


**Result:** training 1,121 masses (587 patients); validation 197 (104 patients); test 378 (201 patients).

## 10. Final checks
Confirms that no patient appears in more than one of the three sets, and shows the label balance.

In [11]:
train_ids = set(final_train_df["patient_id"])
val_ids = set(val_df["patient_id"])
test_ids = set(test_df["patient_id"])

print("Train vs Val:", len(train_ids & val_ids))
print("Train vs Test:", len(train_ids & test_ids))
print("Val vs Test:", len(val_ids & test_ids))

print("\nTraining labels:")
print(final_train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

Train vs Val: 0
Train vs Test: 0
Val vs Test: 0

Training labels:
label
0    589
1    532
Name: count, dtype: int64

Validation labels:
label
1    105
0     92
Name: count, dtype: int64


**Result:** no overlap between any two sets. Training 589 benign / 532 malignant; validation 92 benign / 105 malignant.

## 11. Save the splits
Writes the three tables for notebook 02. `index=False` stops pandas adding a column of row numbers.

In [12]:
import os

output_dir = "/kaggle/working/splits"
os.makedirs(output_dir, exist_ok=True)

final_train_df.to_csv(
    f"{output_dir}/train.csv",
    index=False
)

val_df.to_csv(
    f"{output_dir}/val.csv",
    index=False
)

test_df.to_csv(
    f"{output_dir}/test.csv",
    index=False
)

print("Dataset splits saved successfully!")

Dataset splits saved successfully!


## Summary

| Split | Masses | Patients | Benign | Malignant |
|---|---|---|---|---|
| Training | 1,121 | 587 | 589 | 532 |
| Validation | 197 | 104 | 92 | 105 |
| Test | 378 | 201 | 231 | 147 |
| **Total** | **1,696** | **892** | **912** | **784** |

No patient appears in more than one set. **Next:** notebook 02 attaches this notebook's output and downloads the images.